# v27 — Geneformer V2 고정 임베딩과 아미노산 기반 치환 proxy 분리 실험

팀원 v11_clean.ipynb의 피처 아이디어를 참고합니다. Transformer 전체의 학습/forward가 아닌 유전자 input embedding의 평균·최댓값을 LR에 사용합니다. 치환 비율은 단백질 변화로부터 가정해 계산한 proxy이며 실제 DNA 변이 시그니처로 주장하지 않습니다. base/GF/proxy/둘다 네 가지를 동일 그룹 분할에서 확인합니다. 주 후보는 결과 전 both로 지정하고, 형제 라벨 반전·COAD prior·OOF 비중 탐색은 하지 않습니다. 모든 통계 fit은 각 학습 fold 안에서 수행합니다. test는 마지막 고정 모델 추론에만 사용합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
forbidden_attempts23=[]
def forbid_inference23(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:
  forbidden_attempts23.append(str(args[0]));raise RuntimeError('Inference data access is forbidden in v23')
sys.addaudithook(forbid_inference23)
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if (p/'data/train.csv').is_file()),None)
if REPO is None:raise FileNotFoundError('data/train.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v27';OUT.mkdir(parents=True,exist_ok=True)
# Inference files and submission directory are intentionally not used.
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert train.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)

def normalize_token25(token):
 return re.sub(r'^(p\.)?([ACDEFGHIKLMNPQRSTVWY])([1-9][0-9]*)X$',lambda m:(m[1] or '')+m[2]+m[3]+'*',token)
def canonical_frame25(frame):
 def one(v):
  if not isinstance(v,str) or 'X' not in v:return v
  return ' '.join(normalize_token25(t) for t in v.split())
 return frame.map(one)
def legacy_frame25(frame,mixed=False):
 def one(v):
  if not isinstance(v,str) or '*' not in v:return v
  def conv(m):
   token=m[0]
   if mixed and int(hashlib.sha256(token.encode()).hexdigest()[:8],16)%2:return token
   return token[:-1]+'X'
  return re.sub(r'(?<!\S)(?:p\.)?[ACDEFGHIKLMNPQRSTVWY][1-9][0-9]*\*(?!\S)',conv,v)
 return frame.map(one)

X25_original=train[genes].copy();X25_mixed=legacy_frame25(X25_original,mixed=True);X=canonical_frame25(X25_mixed);assert X.equals(X25_original);groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('TRAIN ONLY',train.shape,'Dev/holdout',len(dev),len(hold),'OUT',OUT,flush=True)

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from sklearn.pipeline import make_pipeline
import gc
AA26="ACDEFGHIKLMNPQRSTVWY"
TYPES26=["missense","silent","nonsense","frameshift","splice","indel","other"]
CFG={"TMB_LOW":5,"TMB_HIGH":30}
gene_pos={g:i for i,g in enumerate(genes)}

from lightgbm import early_stopping


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v27


In [2]:
# P43: HGVS 2016 principles; this is a conservative dataset parser, not a full HGVS validator.
AA16=set('ACDEFGHIKLMNPQRSTVWY');THREE16=dict(zip(['Ala','Arg','Asn','Asp','Cys','Gln','Glu','Gly','His','Ile','Leu','Lys','Met','Phe','Pro','Ser','Thr','Trp','Tyr','Val','Ter','Xaa'],list('ARNDCQEGHILKMFPSTWYV')+['*','X']))
# Each event: key, kind, position, components(position,ref,alt,kind), flags, fs_stop_distance.
def parse_token16(raw):
 s=str(raw).strip();core=s[2:] if s.startswith('p.') else s;uncertain=core.startswith('(') and core.endswith(')');core=core[1:-1] if uncertain else core;core=normalize_token25(core)
 if re.match('^(?:'+'|'.join(THREE16)+')[0-9]+',core):core=re.sub('|'.join(THREE16),lambda m:THREE16[m[0]],core)
 core=normalize_token25(core)
 key=('uncertain|' if uncertain else '')+core;flags=['uncertain'] if uncertain else [];pos=None;parts=();dist=None;kind='unparsed'
 def effect(a,b):
  if a=='*' and b=='*':return 'stop_retained'
  if a=='*':return 'stop_lost' if b in AA16 else 'ambiguous_residue'
  if b=='*':return 'stop_gained' if a in AA16 else 'ambiguous_residue'
  if a not in AA16 or b not in AA16:return 'ambiguous_residue'
  return 'synonymous' if a==b else 'missense'
 m=re.fullmatch(r'([A-Z*])([1-9][0-9]*)([A-Z*])',core)
 if m:
  a,p,b=m.groups();pos=int(p);kind=effect(a,b);parts=((pos,a,b,kind),)
  if 'X' in [a,b]:flags.append('ambiguous_X')
 else:
  m=re.fullmatch(r'([1-9][0-9]*)_([1-9][0-9]*)([A-Z*]+)>([A-Z*]+)',core)
  if m:
   start,end,ref,alt=m.groups();start=int(start);end=int(end);pos=start;span=end-start+1
   if span>0 and span==len(ref)==len(alt) and all(a in AA16 or a=='*' for a in ref+alt):
    parts=tuple((start+j,a,b,effect(a,b)) for j,(a,b) in enumerate(zip(ref,alt)));kind='synonymous_block' if all(a==b for a,b in zip(ref,alt)) else 'block_change';flags.append('resolved_equal_length_block')
   else:kind='complex_range';flags.append('unresolved_range')
  elif 'fs' in core:
   kind='frameshift';m=re.match(r'([A-Z*]+)([1-9][0-9]*)',core);pos=int(m[2]) if m else None;m=re.search(r'fs(?:\*|Ter)([1-9][0-9]*)$',core);dist=int(m[1]) if m else None
  elif 'delins' in core:kind='delins'
  elif 'del' in core:kind='deletion'
  elif 'ins' in core:kind='insertion'
  elif 'dup' in core:kind='duplication'
  elif 'ext' in core:kind='extension'
  elif core=='=':kind='no_change_unspecified'
  elif re.fullmatch(r'[A-Z*]+[1-9][0-9]*',core):kind='incomplete';flags.append('missing_alternate')
  if pos is None:
   # Protein coordinate only: do not mistake fs termination distance for the affected residue.
   m=re.match(r'(?:[A-Z*]+)?([1-9][0-9]*)',core);pos=int(m[1]) if m else None
 return (key,kind,pos,parts,tuple(flags),dist)

def parse_rows16(frame):
 cols=list(frame.columns);cache={};rows=[]
 for row in frame.itertuples(index=False,name=None):
  mutations=[];missing=[];raw_n=0;duplicates=0
  for gene,value in zip(cols,row):
   if value is None or pd.isna(value) or (isinstance(value,str) and not value.strip()):missing.append(gene);continue
   if str(value).strip()=='WT':continue
   tokens=str(value).split();raw_n+=len(tokens);tokens=[normalize_token25(t) for t in tokens];unique=list(dict.fromkeys(tokens));duplicates+=len(tokens)-len(unique);events=[]
   for token in unique:
    if token not in cache:cache[token]=parse_token16(token)
    events.append(cache[token])
   mutations.append((gene,tuple(events)))
  rows.append({'mutations':tuple(mutations),'missing':tuple(missing),'raw_tokens':raw_n,'duplicate_tokens':duplicates})
 return rows

FIXTURES16=[('R132H','missense',132),('R132R','synonymous',132),('R213*','stop_gained',213),('*363*','stop_retained',363),('*213Q','stop_lost',213),('K16Rfs*23','frameshift',16),('1_2FS>FS','synonymous_block',1),('483_484MP>IA','block_change',483),('10_12AAA>DD','complex_range',10),('490del','deletion',490),('p.(Arg123His)','missense',123),('R123X','stop_gained',123),('A391','incomplete',391),('FS','unparsed',None),('G12delinsAA','delins',12)]
for token,expected_kind16,expected_pos16 in FIXTURES16:
 result=parse_token16(token);assert result[1]==expected_kind16 and result[2]==expected_pos16,(token,result)
assert parse_token16('K16Rfs*23')[5]==23
assert parse_token16('1_2FS>FS')[3]==((1,'F','F','synonymous'),(2,'S','S','synonymous'))
assert 'uncertain' in parse_token16('p.(Arg123His)')[4]
assert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[1]['missing']==('G',)
assert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[2]['duplicate_tokens']==1
print('P43 annotation safety fixtures passed:',len(FIXTURES16),flush=True)
# P25-P40: small, explicit gene subsets in the publications, not whole pathways or diagnostic rules.
PANELS16={
 'P25_breast':('TP53','PIK3CA','GATA3','MAP3K1'),
 'P26_colorectal':('APC','TP53','SMAD4','PIK3CA','KRAS','ARID1A','SOX9','FAM123B','POLE','MLH1'),
 'P27_glioma':('IDH1','IDH2','TP53','ATRX','CIC','FUBP1','NOTCH1'),
 'P29_gbm':('EGFR','PDGFRA'),
 'P30_renal':('VHL','PBRM1','SETD2','ARID1A','SMARCA4'),
 'P31_papillary_renal':('MET','SETD2','FH'),
 'P32_ovarian':('TP53','NF1','BRCA1','BRCA2','RB1','CDK12'),
 'P33_uterine':('TP53','PTEN','CTNNB1','PIK3CA','ARID1A','KRAS','ARID5B','POLE'),
 'P34_thyroid':('BRAF','EIF1AX','PPM1D','CHEK2'),
 'P35_prostate':('SPOP','FOXA1','IDH1'),
 'P36_lung_ad':('RIT1','MGA','EGFR','RBM10','NF1','MET','ERBB2'),
 'P37_lung_scc':('TP53','HLA-A','NFE2L2','KEAP1','CDKN2A','RB1'),
 'P38_melanoma':('BRAF','NRAS','HRAS','KRAS','NF1','KIT'),
 'P39_gastric':('PIK3CA','RHOA'),
 'P40_headneck':('PIK3CA','TRAF3','TP53','CDKN2A','HRAS','CASP8','NOTCH1','NSD1','AJUBA','FAT1','NFE2L2')}
print('Literature panel subsets:',len(PANELS16),'(external biological prior; no external patient data)',flush=True)

assert parse_token16("p.Arg123X")==parse_token16("R123*")


P43 annotation safety fixtures passed: 15
Literature panel subsets: 15 (external biological prior; no external patient data)


In [3]:
GENES16=list(genes)
parsed17=parse_rows16(X)

parent17=np.arange(len(train))
def root17(i):
 while parent17[i]!=i:parent17[i]=parent17[parent17[i]];i=parent17[i]
 return i
def join17(a,b):parent17[root17(b)]=root17(a)
canonical17=[hashlib.sha256(repr((tuple(sorted((g,tuple(sorted(set(e[0] for e in ev)))) for g,ev in row['mutations'])),tuple(sorted(row['missing'])))).encode()).hexdigest() for row in parsed17]
for identifiers in [groups,canonical17]:
 seen17={}
 for i,key in enumerate(identifiers):
  if key in seen17:join17(seen17[key],i)
  else:seen17[key]=i
G17=np.array([root17(i) for i in range(len(train))]);overlap17=set(G17[dev])&set(G17[hold]);D17=dev[~np.isin(G17[dev],list(overlap17))]
assert not set(G17[D17])&set(G17[hold])

SHA17=hashlib.sha256((DATA/"train.csv").read_bytes()).hexdigest()
print("Development groups",len(D17),flush=True)


Development groups 4960


In [4]:
import urllib.request,pickle
GF_MANIFEST27={'repo': 'ctheodoris/Geneformer', 'revision': '1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5', 'model': 'Geneformer-V2-104M', 'embedding_key': 'bert.embeddings.word_embeddings.weight', 'shape': [20275, 768], 'matched': 4377, 'genes': 4384, 'coverage': 0.9984032846715328, 'license': 'apache-2.0', 'files': {'model.safetensors': 'fff5cba29ddd8792991fa77b4872246fbe548a178cebda3775cdc72b67780e7f', 'gene_name_id_dict_gc104M.pkl': 'fabfa0c2f49c598c59ae432a32c3499a5908c033756c663b5e0cddf58deea8e1', 'gene_embedding_table.npz': '30cef571bc6a954bb19d25297d64e61e06dc43e08e07b4cc3a576a10067d43c8', 'config.json': '467d4492f0dd53b4d60afffe20812db484ca1cf9fdbeb6a6e060e93564f70859', 'README.md': '56d6e570b349cbedae9a54634421c94e7af8ea467ce0fdd79193372ae3cbdbd8', 'token_dictionary_gc104M.pkl': '67c445f4385127adfc48dcc072320cd65d6822829bf27dd38070e6e787bc597f'}, 'table_float32_sha256': 'dd8c2ba98e3da00773e82d96c23520e7a760ad7a717d1bf55a8c9a055a90dd75'}
def sha_file27(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()
class PrimitiveDictionary27(pickle.Unpickler):
 def find_class(self,module,name):raise pickle.UnpicklingError('Only primitive dictionaries allowed')
def load_embedding27():
 folder=OUT/'pretrained';folder.mkdir(parents=True,exist_ok=True);tablepath=folder/'gene_embedding_table.npz'
 if tablepath.exists():
  with np.load(tablepath,allow_pickle=False) as q:
   assert q['genes'].tolist()==list(genes);table=q['table'].copy()
 else:
  mapping={'model.safetensors':'Geneformer-V2-104M/model.safetensors','config.json':'Geneformer-V2-104M/config.json','gene_name_id_dict_gc104M.pkl':'geneformer/gene_name_id_dict_gc104M.pkl','token_dictionary_gc104M.pkl':'geneformer/token_dictionary_gc104M.pkl'}
  for name,remote in mapping.items():
   dest=folder/name
   if not dest.exists():urllib.request.urlretrieve('https://huggingface.co/ctheodoris/Geneformer/resolve/'+GF_MANIFEST27['revision']+'/'+remote,dest)
   assert sha_file27(dest)==GF_MANIFEST27['files'][name]
  with (folder/'gene_name_id_dict_gc104M.pkl').open('rb') as f:names=PrimitiveDictionary27(f).load()
  with (folder/'token_dictionary_gc104M.pkl').open('rb') as f:tokens=PrimitiveDictionary27(f).load()
  with (folder/'model.safetensors').open('rb') as f:
   size=int.from_bytes(f.read(8),'little');assert 0<size<10_000_000;meta=json.loads(f.read(size))[GF_MANIFEST27['embedding_key']];assert meta['dtype']=='F32';a,b=meta['data_offsets'];assert 0<=a<b and 8+size+b<= (folder/'model.safetensors').stat().st_size;f.seek(8+size+a);weights=np.frombuffer(f.read(b-a),np.float32).reshape(meta['shape'])
  table=np.zeros((len(genes),weights.shape[1]),np.float32)
  for j,g in enumerate(genes):
   if g in names and names[g] in tokens:table[j]=weights[int(tokens[names[g]])]
  np.savez_compressed(tablepath,genes=np.array(genes),table=table)
 assert table.shape==(len(genes),768) and np.isfinite(table).all() and hashlib.sha256(table.tobytes()).hexdigest()==GF_MANIFEST27['table_float32_sha256']
 return table
GF_EMB=load_embedding27();(OUT/'pretrained/manifest.json').write_text(json.dumps(GF_MANIFEST27,indent=2));print('Geneformer fixed gene table',GF_EMB.shape,'coverage',GF_MANIFEST27['coverage'],flush=True)


Geneformer fixed gene table (4384, 768) coverage 0.9984032846715328


In [5]:
KINDS=["missense","silent","nonsense","frameshift","inframe","other"]
import scipy.sparse as sp

TRUNC = {"nonsense", "frameshift"}       # 절단형 변이
NO_HS = {"silent", "other"}              # 핫스팟에서 제외 (기능 변화가 없거나 해석 불가)

# 변이 시그니처 (도메인 지식): 발암 원인마다 남기는 DNA 치환 흔적이 다르다.
#   자외선(SKCM) → C>T, 흡연(LUAD·LUSC) → C>A, APOBEC(BLCA·CESC) → C>T·C>G
# 데이터에는 DNA가 없고 아미노산 변화만 있으므로, 표준 유전 암호표(64개 코돈)로
# "이 아미노산 변화를 만들 수 있는 염기 하나 치환"을 모두 찾아 같은 비중으로 나눠 센다.
# 치환은 관례대로 피리미딘(C·T) 기준 6종으로 접는다 (G>A는 C>T와 같은 것으로 셈).
SBS6 = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
_B = "TCAG"
CODON = {a + b + c: aa for (a, b, c), aa in zip(((a, b, c) for a in _B for b in _B for c in _B),
                                                "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG")}
_COMP = {"A": "T", "C": "G", "G": "C", "T": "A"}


def _sig_table():
    """(바뀌기 전 아미노산, 바뀐 후) → SBS6 비중 벡터(합 1). 염기 하나로 만들 수 없는 변화는 없음."""
    acc = {}
    for cod, a in CODON.items():
        for i in range(3):
            for x in "ACGT":
                if x == cod[i]:
                    continue
                b = CODON[cod[:i] + x + cod[i + 1:]]
                r, t = cod[i], x
                if r in "AG":                       # 퓨린 기준이면 상보 가닥(피리미딘 기준)으로 접기
                    r, t = _COMP[r], _COMP[t]
                acc.setdefault((a, b), np.zeros(6))[SBS6.index(f"{r}>{t}")] += 1
    return {k: v / v.sum() for k, v in acc.items()}


SIG = _sig_table()


class FeatureBuilder:
    def __init__(self, genes, min_gene=5, min_trunc=3, min_hs=3, use_aa=False, min_aa=20, gf_emb=None, use_sig=False):
        self.genes = genes
        self.min_gene, self.min_trunc, self.min_hs = min_gene, min_trunc, min_hs
        self.use_aa, self.min_aa = use_aa, min_aa
        self.use_sig = use_sig
        self.gf_emb = gf_emb      # (유전자 수, d) 사전학습 임베딩 표. 고정값이라 fit에서 배울 것이 없음

    @staticmethod
    def _keys(row):
        """한 행 → (변이 유전자 집합, 절단형 유전자 집합, 핫스팟 키 집합)"""
        g = {e[0] for e in row}
        t = {e[0] for e in row if e[1] in TRUNC}
        h = {(e[0], f"{e[2]}{e[3]}") for e in row if e[1] not in NO_HS and e[3] >= 0}
        return g, t, h

    def fit(self, rows, ids):
        """학습 부분의 행만으로 단어장을 정한다. ids는 누수 검사용으로 기록만 한다."""
        self.fit_ids = np.asarray(ids)
        cg, ct, ch, caa = Counter(), Counter(), Counter(), Counter()
        for row in rows:
            g, t, h = self._keys(row)
            cg.update(g); ct.update(t); ch.update(h)
            if self.use_aa:
                caa.update((e[2], e[4]) for e in row if e[1] == "missense")
        self.g_idx = {k: i for i, k in enumerate(sorted(k for k, v in cg.items() if v >= self.min_gene))}
        self.t_idx = {k: i for i, k in enumerate(sorted(k for k, v in ct.items() if v >= self.min_trunc))}
        self.h_idx = {k: i for i, k in enumerate(sorted(k for k, v in ch.items() if v >= self.min_hs))}
        self.aa_idx = {k: i for i, k in enumerate(sorted(k for k, v in caa.items() if v >= self.min_aa))}
        self.burden_names = ["log_n_genes", "n_events"] + [f"cnt_{k}" for k in KINDS] + [f"frac_{k}" for k in KINDS]
        G = self.genes
        self.names = ([f"g_{G[k]}" for k in self.g_idx] + [f"t_{G[k]}" for k in self.t_idx] +
                      [f"hs_{G[k[0]]}|{k[1]}" for k in self.h_idx] + self.burden_names +
                      [f"aa_{a}>{b}" for a, b in self.aa_idx] +
                      ([f"sig_{k}" for k in SBS6] if self.use_sig else []))
        self.n_lgb = len(self.names)   # LightGBM은 앞의 이 열들만 쓴다 (GF 열은 LR 전용)
        if self.gf_emb is not None:
            d = self.gf_emb.shape[1]
            self.names += [f"gf_mean_{i}" for i in range(d)] + [f"gf_max_{i}" for i in range(d)]
        return self

    def transform(self, rows):
        """단어장대로 희소 행렬(float32)로 바꾼다. 각 행은 그 행만 보고 변환된다."""
        o_t = len(self.g_idx); o_h = o_t + len(self.t_idx); o_b = o_h + len(self.h_idx)
        o_a = o_b + len(self.burden_names)
        R, C, V = [], [], []
        for r, row in enumerate(rows):
            g, t, h = self._keys(row)
            cols = ([self.g_idx[k] for k in g if k in self.g_idx] +
                    [o_t + self.t_idx[k] for k in t if k in self.t_idx] +
                    [o_h + self.h_idx[k] for k in h if k in self.h_idx])
            R += [r] * len(cols); C += cols; V += [1.0] * len(cols)
            cnt = Counter(e[1] for e in row); n_ev = len(row)
            b = [np.log1p(len(g)), n_ev] + [cnt[k] for k in KINDS] + [cnt[k] / (n_ev + 1) for k in KINDS]
            R += [r] * len(b); C += list(range(o_b, o_b + len(b))); V += b
            if self.aa_idx:
                ca = Counter((e[2], e[4]) for e in row if e[1] == "missense")
                for k, v in ca.items():
                    if k in self.aa_idx:
                        R.append(r); C.append(o_a + self.aa_idx[k]); V.append(v / (cnt["missense"] + 1))
            if self.use_sig:
                o_s = o_a + len(self.aa_idx)
                sv = sum((SIG[(e[2], "*" if e[4] == "X" else e[4])] for e in row
                          if e[1] in ("missense", "nonsense", "silent") and (e[2], "*" if e[4] == "X" else e[4]) in SIG),
                         np.zeros(6))
                if sv.sum() > 0:
                    sv = sv / sv.sum()
                R += [r] * 6; C += list(range(o_s, o_s + 6)); V += sv.tolist()
            if self.gf_emb is not None:
                gi = sorted({e[0] for e in row if e[1] != "silent"})
                if gi:          # 변이가 없으면 0으로 둔다
                    M = self.gf_emb[gi]
                    b = np.concatenate([M.mean(0), M.max(0)])
                    o_g = len(self.names) - len(b)
                    R += [r] * len(b); C += list(range(o_g, o_g + len(b))); V += b.tolist()
        X = sp.csr_matrix((np.asarray(V, np.float32), (R, C)), shape=(len(rows), len(self.names)))
        X.eliminate_zeros()
        return X
assert CODON["ATG"]=="M" and CODON["TGA"]=="*" and CODON["AAA"]=="K"
assert all(np.isclose(v.sum(),1) for v in SIG.values())


In [6]:
def rows_for27(frame):
 raw=parse_rows16(canonical_frame25(frame.loc[:,genes]));result=[];gp={g:i for i,g in enumerate(genes)}
 for row in raw:
  events=[]
  for g,evs in row['mutations']:
   for key,kind,pos,parts,flags,dist in evs:
    typ={'synonymous':'silent','synonymous_block':'silent','stop_gained':'nonsense','deletion':'inframe','insertion':'inframe','duplication':'inframe','delins':'inframe'}.get(kind,kind if kind in {'missense','frameshift'} else 'other')
    ref=parts[0][1] if parts else (m[0] if (m:=re.match(r'[ACDEFGHIKLMNPQRSTVWY]',key.removeprefix('uncertain|'))) else '')
    alt=parts[0][2] if len(parts)==1 else ''
    events.append((gp[g],typ,ref,pos if pos is not None else -1,alt))
  result.append(events)
 return result

def new_tree27(rounds):
 return LGBMClassifier(objective='multiclass',class_weight='balanced',num_leaves=15,learning_rate=.05,colsample_bytree=.3,n_estimators=rounds,n_jobs=2,random_state=27027,verbosity=-1)
def fit_tree27(indices,sig,seed=27027,rounds=None):
 if rounds is None:
  aa,bb=next(StratifiedGroupKFold(5,shuffle=True,random_state=seed).split(np.zeros(len(indices)),yi[indices],G17[indices]));it,iv=indices[aa],indices[bb];assert not set(G17[it])&set(G17[iv]);fb=FeatureBuilder(genes,5,3,3,gf_emb=None,use_sig=sig).fit([parsed27[i] for i in it],it);a=fb.transform([parsed27[i] for i in it]);b=fb.transform([parsed27[i] for i in iv]);tree=new_tree27(3000)
  print('V27 inner tree',sig,len(it),a.shape[1],flush=True);tree.fit(a,yi[it],eval_set=[(b,yi[iv])],callbacks=[early_stopping(100,verbose=False)]);rounds=int(tree.best_iteration_);del tree,fb,a,b;gc.collect()
 fb=FeatureBuilder(genes,5,3,3,gf_emb=None,use_sig=sig).fit([parsed27[i] for i in indices],indices);tree=new_tree27(int(rounds));print('V27 refit tree',sig,len(indices),len(fb.names),rounds,flush=True);tree.fit(fb.transform([parsed27[i] for i in indices]),yi[indices]);return fb,tree,int(rounds)
class Model27:
 def __init__(self,sig=True,gf=True):self.sig=sig;self.gf=gf;self.genes_=list(genes);self.classes_=labels.copy()
 def fit(self,indices,tree_bundle=None,rounds=None):
  self.treepre,self.tree,self.rounds_=tree_bundle if tree_bundle is not None else fit_tree27(indices,self.sig,rounds=rounds)
  self.lrpre=FeatureBuilder(genes,5,3,3,gf_emb=GF_EMB if self.gf else None,use_sig=self.sig).fit([parsed27[i] for i in indices],indices)
  self.lr=make_pipeline(MaxAbsScaler(),LogisticRegression(C=.1,class_weight='balanced',max_iter=1000));print('V27 fit LR',self.sig,self.gf,len(indices),len(self.lrpre.names),flush=True);self.lr.fit(self.lrpre.transform([parsed27[i] for i in indices]),yi[indices]);return self
 def predict_parts(self,rows):return self.tree.predict_proba(self.treepre.transform(rows)),self.lr.predict_proba(self.lrpre.transform(rows))
 def predict_rows(self,rows):
  a,b=self.predict_parts(rows);p=.5*a+.5*b;assert p.shape==(len(rows),len(labels)) and np.isfinite(p).all() and np.allclose(p.sum(1),1);return p
 def predict_frame(self,frame):return self.predict_rows(rows_for27(frame.loc[:,self.genes_]))

parsed27=rows_for27(X)
assert rows_for27(X.iloc[:12])==rows_for27(legacy_frame25(X.iloc[:12]))


In [7]:
REFERENCE_SPLITS27=['06c06464902eb31794a47a2ba66fbddb68356ca00f496ca48467bed9ce5c57e8', 'ee3e3488ac4ee01532dfd255f21afe072913d5016f76cb0f9809a3af3b86e7f0', 'b19d1be892ad77c9ef9b6f9712fc569695526bd7cce7b36f9b05eb2512286a0d']
PROTOCOL27={'version':27,'main_candidate':'both','variants':{'base':[False,False],'geneformer':[False,True],'proxy':[True,False],'both':[True,True]},'source':'user-supplied v11_clean.ipynb (Minjun), cells11/13; implemented with conservative v25 parser and group-safe fitting','geneformer':'fixed V2-104M input embedding, mean/max non-silent mutated gene pools, LR only, no Transformer forward or fine-tuning','signature':'uniform-codon AA-derived six substitution proxy; NOT measured DNA/SBS96 signatures or proven mutational process','xgb_lr':False,'lgb_lr_weights':[.5,.5],'lr_C':.1,'lgb':{'leaves':15,'lr':.05,'feature_fraction':.3,'max_inner_rounds':3000,'patience':100},'outer_folds':3,'outer_seed':23033,'inner_group_folds':5,'early_stop':'inner train-only feature vocabulary; inner group validation; then outer-training vocabulary and tree refit','test_for_research':False,'external_knowledge':'official pinned pretrained non-cancer expression model; external patient files/competition test not used for fitting; pretraining identity overlap not independently audited','provenance':'legacy-X assumption retained from v25 after earlier test notation inspection; repeated development cohort','forbidden_rules':['twin label flip','COAD prior correction','OOF blend weight search']}
(OUT/'protocol_v27.json').write_text(json.dumps(PROTOCOL27,indent=2));results27=[];oof27={key:np.zeros((len(D17),NC)) for key in PROTOCOL27['variants']};main_rounds27=[];f27=lambda t,p:float(f1_score(t,p.argmax(1),labels=np.arange(NC),average='macro',zero_division=0))
reference_scores27=[0.4813030372414948,0.4973466832302821,0.5006005643662187]
assert SHA17=='92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab'
for fold,(aa,bb) in enumerate(StratifiedGroupKFold(3,shuffle=True,random_state=23033).split(X.iloc[D17],yi[D17],G17[D17])):
 tr,va=D17[aa],D17[bb];assert not set(G17[tr])&set(G17[va]);assert not set(G17[tr])&set(G17[hold]);assert hashlib.sha256(tr.tobytes()+va.tobytes()).hexdigest()==REFERENCE_SPLITS27[fold]
 current=[parsed27[i] for i in va]
 for sig in [False,True]:
  bundle=fit_tree27(tr,sig,seed=27027+fold)
  for gf in [False,True]:
   name={ (False,False):'base',(False,True):'geneformer',(True,False):'proxy',(True,True):'both'}[(sig,gf)];model27=Model27(sig,gf).fit(tr,tree_bundle=bundle);ap,bp=model27.predict_parts(current);p=.5*ap+.5*bp;oof27[name][bb]=p
   probe=X.iloc[va[:12]];assert np.allclose(model27.predict_frame(probe),model27.predict_frame(legacy_frame25(probe)),atol=1e-7)
   record={'fold':fold,'candidate':name,'tree_features':len(model27.treepre.names),'lr_features':len(model27.lrpre.names),'rounds':model27.rounds_,'macro_f1':f27(yi[va],p),'accuracy':float(accuracy_score(yi[va],p.argmax(1))),'tree_macro_f1':f27(yi[va],ap),'lr_macro_f1':f27(yi[va],bp),'v25_macro_f1':reference_scores27[fold],'delta_vs_v25':f27(yi[va],p)-reference_scores27[fold]};results27.append(record);pd.DataFrame(results27).to_csv(OUT/'group_cv_results.csv',index=False);np.savez_compressed(OUT/f'fold{fold}_{name}.npz',tr=tr,va=va,probabilities=p,tree=ap,lr=bp)
   print('V27 RESULT',record,flush=True)
   if name=='both':main_rounds27.append(model27.rounds_)
   del model27;gc.collect()
  del bundle;gc.collect()
table27=pd.DataFrame(results27);means27=table27.groupby('candidate')[['macro_f1','accuracy','delta_vs_v25']].mean();main27=table27[table27.candidate=='both'];summary27={'version':27,'results':means27.to_dict('index'),'main_candidate':'both','formal_improvement_gate':'mean F1 +0.003 vs v25 and every fold >=-0.001','improvement_gate_passed':bool(main27.delta_vs_v25.mean()>=.003 and main27.delta_vs_v25.min()>=-.001),'model_saved':False,'public_score':None,'train_sha256':SHA17,'test_for_research':False,'geneformer_coverage':GF_MANIFEST27['coverage']};(OUT/'verification_v27.json').write_text(json.dumps(summary27,indent=2));np.savez_compressed(OUT/'oof_v27.npz',dev=D17,classes=labels,**oof27)
# Predeclared both model saved as experimental even if the comparison fails; no outer-best recipe selection.
rounds27=int(np.median(main_rounds27));print('V27 FULL TRAIN',rounds27,flush=True);final27=Model27(True,True).fit(np.arange(len(train)),rounds=rounds27);final27.training_data_sha256_=SHA17;final27.training_protocol_=PROTOCOL27;final27.geneformer_provenance_=GF_MANIFEST27
path27=OUT/'cancer_model_v27_geneformer_proxy_experimental.pkl'
with path27.open('wb') as f:cloudpickle.dump(final27,f)
with path27.open('rb') as f:reload27=cloudpickle.load(f)
assert np.allclose(final27.predict_frame(X.iloc[:8]),reload27.predict_frame(legacy_frame25(X.iloc[:8])),atol=1e-7)
pd.DataFrame({'model':'lr','feature':final27.lrpre.names}).to_csv(OUT/'selected_variables.csv',index=False);assert not forbidden_attempts23
summary27.update(model_saved=True,model_sha256=hashlib.sha256(path27.read_bytes()).hexdigest(),final_tree_features=len(final27.treepre.names),final_lr_features=len(final27.lrpre.names),final_rounds=rounds27,legacy_notation_reload_passed=True);(OUT/'verification_v27.json').write_text(json.dumps(summary27,indent=2));print('V27 COMPLETE',json.dumps(summary27,indent=2),flush=True)


V27 inner tree False 2645 5290
V27 refit tree False 3306 6241 102
V27 fit LR False False 3306 6241
V27 RESULT {'fold': 0, 'candidate': 'base', 'tree_features': 6241, 'lr_features': 6241, 'rounds': 102, 'macro_f1': 0.39521681472540643, 'accuracy': 0.41777509068923824, 'tree_macro_f1': 0.3880164455222158, 'lr_macro_f1': 0.38529480967663676, 'v25_macro_f1': 0.4813030372414948, 'delta_vs_v25': -0.08608622251608838}
V27 fit LR False True 3306 7777
V27 RESULT {'fold': 0, 'candidate': 'geneformer', 'tree_features': 6241, 'lr_features': 7777, 'rounds': 102, 'macro_f1': 0.40978932323723427, 'accuracy': 0.4322853688029021, 'tree_macro_f1': 0.3880164455222158, 'lr_macro_f1': 0.3735578646315179, 'v25_macro_f1': 0.4813030372414948, 'delta_vs_v25': -0.07151371400426054}
V27 inner tree True 2645 5296
V27 refit tree True 3306 6247 112
V27 fit LR True False 3306 6247
V27 RESULT {'fold': 0, 'candidate': 'proxy', 'tree_features': 6247, 'lr_features': 6247, 'rounds': 112, 'macro_f1': 0.44342487514960116, 

/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X

## 최종 추론
사전 지정한 both 전체 train 실험 모델로 최종 예측만 수행합니다. 연구 프로세스의 test 접근 금지를 유지하고 별도 자식 프로세스에서 실행합니다. 개선 기준 미달이면 새 최고 모델로 추천하지 않습니다.

In [8]:
# 별도 프로세스에서 고정 모델로 최종 test 예측만 수행합니다.
from pathlib import Path
import subprocess,sys
roots27=[Path.cwd(),*Path.cwd().parents]
repo27=next((p for p in roots27 if (p/'data/train.csv').is_file()),None)
if repo27 is None:raise FileNotFoundError('data 폴더가 필요합니다.')
work27=repo27/'models'/'혜진' if (repo27/'models'/'혜진').is_dir() else repo27
program27="import sys,os,json,hashlib,cloudpickle,numpy as np,pandas as pd\nfrom pathlib import Path\nfrom threadpoolctl import threadpool_limits\nthreadpool_limits(2)\ndef guard(e,a):\n if e=='open' and a and isinstance(a[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(a[0])).name=='train.csv':raise RuntimeError('Final inference does not read train.csv')\nsys.addaudithook(guard)\nmp,tp,sp,op=map(Path,sys.argv[1:5]);proof=json.loads((mp.parent/'verification_v27.json').read_text());assert proof['model_saved']\nassert hashlib.sha256(mp.read_bytes()).hexdigest()==proof['model_sha256']\nwith mp.open('rb') as f:model=cloudpickle.load(f)\nt=pd.read_csv(tp);s=pd.read_csv(sp);assert t.ID.is_unique and s.ID.is_unique and len(t)==len(s) and set(t.ID)==set(s.ID)\nx=t.loc[:,model.genes_];p=model.predict_frame(x);assert p.shape==(len(t),26) and np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(1),1)\nidx=[10,3,500,0];assert np.allclose(model.predict_frame(x.iloc[idx]),p[idx],atol=1e-6)\ns.SUBCLASS=s.ID.map(pd.Series(model.classes_[p.argmax(1)],index=t.ID));assert not s.isna().any().any()\nop.parent.mkdir(parents=True,exist_ok=True)\npayload=s.to_csv(index=False).encode()\nif op.exists():assert op.read_bytes()==payload,'Existing v27 submission differs; preserve it.'\nelse:op.write_bytes(payload)\nassert pd.read_csv(op).ID.equals(s.ID)\nreport={'version':27,'rows':len(s),'model_sha256':proof['model_sha256'],'csv_sha256':hashlib.sha256(op.read_bytes()).hexdigest(),'phase':'final inference only','model_fitting':False,'test_used_for_validation':False,'public_score':None,'batch_invariance':True}\n(mp.parent/'submission_verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2));print(json.dumps(report,ensure_ascii=False,indent=2))\n"
args27=[str(work27/'artifacts/v27/cancer_model_v27_geneformer_proxy_experimental.pkl'),str(repo27/'data/test.csv'),str(repo27/'data/sample_submission.csv'),str(repo27/'submission/혜진_submission_v27_geneformer_proxy_experimental.csv')]
run27=subprocess.run([sys.executable,'-c',program27,*args27],capture_output=True,text=True)
if run27.returncode:raise RuntimeError(run27.stderr)
print(run27.stdout)


{
  "version": 27,
  "rows": 2546,
  "model_sha256": "053d1cfb8ea667a50cc0d43c817e2a2aa1a8279ad8bb098eaa3cb74f9124d604",
  "csv_sha256": "06ff264ba8e59c1d002dd32e502d9fe2bc9151f119f29dde21c680e4d397aebc",
  "phase": "final inference only",
  "model_fitting": false,
  "test_used_for_validation": false,
  "public_score": null,
  "batch_invariance": true
}